In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import rasterio

drive.mount('/content/drive')

class EDSR(tf.keras.Model):
    def __init__(self, scale=6):
        super().__init__()
        self.scale = scale
        self.conv1 = None
        self.res_blocks = []
        self.upscale = None

    def build(self, input_shape):
        """Явное построение модели с инициализацией слоев"""
        # Первый сверточный слой
        self.conv1 = tf.keras.layers.Conv2D(
            256, 3, padding='same',
            kernel_initializer='he_normal'
        )

        # Residual блоки
        self.res_blocks = [self.ResBlock() for _ in range(16)]

        # Блок апсемплинга
        self.upscale = tf.keras.Sequential([
            tf.keras.layers.Conv2D(256 * (self.scale**2), 3, padding='same'),
            tf.keras.layers.Lambda(lambda x: tf.nn.depth_to_space(x, self.scale)),
            tf.keras.layers.Conv2D(1, 3, padding='same')
        ])

        super().build(input_shape)

    class ResBlock(tf.keras.layers.Layer):
        def build(self, input_shape):
            """Построение residual блока"""
            self.conv1 = tf.keras.layers.Conv2D(
                256, 3, padding='same',
                activation='relu',
                kernel_initializer='he_normal'
            )
            self.conv2 = tf.keras.layers.Conv2D(
                256, 3, padding='same',
                kernel_initializer='he_normal'
            )
            self.built = True

        def call(self, x):
            return x + self.conv2(self.conv1(x))

    def call(self, inputs):
        x = self.conv1(inputs)
        for block in self.res_blocks:
            x = block(x)
        return self.upscale(x)

def plot_elevation(input_path, weights_path):
    # Инициализация и построение модели
    model = EDSR(scale=6)

    try:
        # Явное указание входной формы
        model.build((None, 42, 42, 1))
        model.load_weights(weights_path)
        print("✓ Модель успешно загружена")
    except Exception as e:
        print(f"✗ Ошибка: {str(e)}")
        return

    # Обработка данных и визуализация
    try:
        with rasterio.open(input_path) as src:
            data = src.read(1).astype(np.float32)

            # Нормализация данных
            valid_data = data[data != src.nodata]
            normalized = (data - np.min(valid_data)) / (np.max(valid_data) - np.min(valid_data))

            # Подготовка входных данных
            lr_input = tf.image.resize(
                normalized[np.newaxis, ..., np.newaxis],
                (42, 42),
                method='area'
            )

            # Прогнозирование
            prediction = model.predict(lr_input)[0, ..., 0]

            # Визуализация
            plt.figure(figsize=(12, 6))
            plt.imshow(prediction, cmap='terrain')
            plt.title('Карта высот (разрешение 5м)', pad=20, fontsize=14)
            plt.xlabel('Широта', fontsize=12)
            plt.ylabel('Долгота', fontsize=12)
            plt.colorbar(label='Высота (м)', orientation='vertical', pad=0.02)
            plt.tight_layout()
            plt.show()

    except Exception as e:
        print(f"Ошибка обработки данных: {str(e)}")


# Пример использования
plot_elevation(
    input_path='/content/drive/MyDrive/JAXA_DATA/jaxa_export.tif',
#    output_path='/content/drive/MyDrive/high_res_dem.tif',
    weights_path='/content/drive/MyDrive/dem_sr_final_2.weights.h5'
)

In [ ]:
!pip install tensorflow==2.18.0 numpy==1.26.4
!pip install rasterio==1.3.9 earthengine-api==0.1.408
!pip install google-cloud-storage